# SocialChoice 03 : Methodes de Vote et Paradoxes (twin C# .NET)

> **Twin C# .NET de `GameTheory/SocialChoice/03-Voting-Methods.ipynb`** (marathon parite #4956). Kernel `.net-csharp`. Ce notebook compagnon du **notebook 02 (Lean, preuves formelles)** fournit les **implementations C# from-scratch** des methodes de vote (Prong B, EPIC #3801). Le twin Python utilise numpy/matplotlib/networkx ; les visualisations sont remplacees par un rendu ASCII autonome.

## Objectifs d'apprentissage

1. **Modeliser** un profil de preferences collectives (classements individuels)
2. **Implementer** les regles de vote : Pluralite, Borda, Copeland, Condorcet, IRV
3. **Illustrer** les paradoxes : cycle de Condorcet, divergence des regles, theoreme de Sen
4. **Simuler** empiriquement les violations de IIA (theoreme d'Arrow)
5. **Comprendre** le theoreme de l'electeur median (preferences unimodales)

In [1]:
// SocialChoice 03 : Methodes de Vote et Paradoxes -- twin C# de 03-Voting-Methods
// Prong B (#3801) : implementations from-scratch des regles de vote (Pluralite, Borda,
// Copeland, Condorcet, IRV). Le twin Python (03-Voting-Methods.ipynb) est le compagnon
// du notebook 02 (Lean, preuves formelles) ; ce twin execute les memes algorithmes en C#.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Culture invariante : separateur decimal "." (parite de sortie avec le twin Python).
// Les 3 setters sont necessaires : .NET Interactive evalue les cellules sur des threads
// du pool distincts (CurrentCulture seul ne persiste pas cross-cell).
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

Console.WriteLine("Configuration OK : SocialChoice 03 - Methodes de Vote (twin C# .NET 9)");


Configuration OK : SocialChoice 03 - Methodes de Vote (twin C# .NET 9)


In [2]:
// === Section 1 : Profil de preferences ===
// Un profil = liste des classements (du meilleur au pire) de chaque votant.
// Equivalent C# de la classe Profile du twin Python / de la structure Lean Profile (SC-02).

static class Vote
{
    // Le votant i prefere-t-il x a y ? (x et y sont des indices d'alternatives)
    public static bool Prefers(List<List<string>> profile, int voter, string x, string y)
        => profile[voter].IndexOf(x) < profile[voter].IndexOf(y);

    // Majorite des votants prefere-t-elle x a y ?
    public static bool MajorityPrefers(List<List<string>> profile, string x, string y)
    {
        int xCount = profile.Count(p => p.IndexOf(x) < p.IndexOf(y));
        return xCount > profile.Count / 2.0;
    }

    // Resultat du duel majoritaire entre x et y : x, y, ou null (egalite)
    public static string PairwiseMajority(List<List<string>> profile, string x, string y)
    {
        int xWins = profile.Count(p => p.IndexOf(x) < p.IndexOf(y));
        int yWins = profile.Count - xWins;
        if (xWins > yWins) return x;
        if (yWins > xWins) return y;
        return null;
    }
}

// Profil de Condorcet (3 votants, cycle A>B>C / B>C>A / C>A>B)
var condorcetProfile = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "B", "C", "A" },
    new() { "C", "A", "B" },
};

static void PrintProfile(string title, List<List<string>> profile)
{
    Console.WriteLine(title);
    Console.WriteLine(new string('=', 40));
    for (int i = 0; i < profile.Count; i++)
        Console.WriteLine($"  Votant {i + 1}: {string.Join(" > ", profile[i])}");
}

PrintProfile("PROFIL DE CONDORCET", condorcetProfile);
Console.WriteLine($"\nPareto(A, B) ici ? {Vote.PairwiseMajority(condorcetProfile, "A", "B")} (A bat B en duel)");


PROFIL DE CONDORCET


  Votant 1: A > B > C


  Votant 2: B > C > A


  Votant 3: C > A > B



Pareto(A, B) ici ? A (A bat B en duel)


## 1. Profil de preferences et paradoxe de Condorcet

Un **profil** decrit les classements (du meilleur au pire) de chaque votant. La structure `Profile` est l'equivalent C# de la structure Lean du notebook SC-02. Le **paradoxe de Condorcet** : avec 3 alternatives et des preferences cycliques, les duels majoritaires forment un cycle (A bat B, B bat C, C bat A) -- aucun gagnant n'emerge.

In [3]:
// === Section 1 (suite) : Paradoxe de Condorcet et cycles ===
// On calcule tous les duels pairwise. S'il y a un cycle (A bat B, B bat C, C bat A),
// aucun gagnant de Condorcet n'existe.

static void CheckCondorcetCycle(List<List<string>> profile, List<string> alternatives)
{
    Console.WriteLine("Resultats pairwise :");
    for (int i = 0; i < alternatives.Count; i++)
        for (int j = i + 1; j < alternatives.Count; j++)
        {
            string x = alternatives[i], y = alternatives[j];
            string winner = Vote.PairwiseMajority(profile, x, y);
            Console.WriteLine($"  {x} vs {y}: {(winner ?? "egalite")}");
        }
}

CheckCondorcetCycle(condorcetProfile, new List<string> { "A", "B", "C" });
Console.WriteLine("\n=> CYCLE : A bat B, B bat C, C bat A ! (chacun 2-1)");

// Visualisation ASCII du cycle (remplace networkx du twin Python)
Console.WriteLine("\nGraphe oriente des duels (cycle) :");
Console.WriteLine("    A");
Console.WriteLine("   ^ \\");
Console.WriteLine("   |  v");
Console.WriteLine("   C <- B");
Console.WriteLine("   A > B, B > C, C > A  (cycle de longueur 3)");


Resultats pairwise :


  A vs B: A


  A vs C: C


  B vs C: B



=> CYCLE : A bat B, B bat C, C bat A ! (chacun 2-1)


...

    A


   ^ \


   |  v


   C <- B


   A > B, B > C, C > A  (cycle de longueur 3)


### Cycle de Condorcet

On enumere tous les duels pairwise. Le cycle ( chacun 2-1) montre qu'aucun candidat ne bat tous les autres : il n'y a **pas de gagnant de Condorcet**.

In [4]:
// === Section 2 : Gagnant de Condorcet (version generale) ===
// Un candidat est gagnant de Condorcet s'il bat tous les autres en duel.
// Retourne null s'il y a un cycle (aucun gagnant).

static string CondorcetWinner(List<List<string>> profile, List<string> alternatives)
{
    foreach (var candidate in alternatives)
    {
        bool beatsAll = true;
        foreach (var other in alternatives)
        {
            if (other == candidate) continue;
            if (Vote.PairwiseMajority(profile, candidate, other) != candidate)
            {
                beatsAll = false;
                break;
            }
        }
        if (beatsAll) return candidate;
    }
    return null;
}

// Profil cyclique : aucun gagnant
Console.WriteLine($"Profil cyclique   : gagnant = {CondorcetWinner(condorcetProfile, new() { "A", "B", "C" }) ?? "AUCUN (cycle)"}");

// Profil avec un gagnant de Condorcet (A)
var profileWithWinner = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "A", "C", "B" },
    new() { "B", "A", "C" },
    new() { "C", "A", "B" },
    new() { "A", "B", "C" },
};
Console.WriteLine($"Profil a gagnant  : gagnant = {CondorcetWinner(profileWithWinner, new() { "A", "B", "C" })}");


Profil cyclique   : gagnant = AUCUN (cycle)


Profil a gagnant  : gagnant = A


## 2. Gagnant de Condorcet

Un candidat est **gagnant de Condorcet** s'il bat tous les autres en duel majoritaire. Sur un profil cyclique, il n'existe pas (retour null). Sur un profil favorable, un candidat (A) emerge.

In [5]:
// === Section 3 : Regles de vote positionnelles ===
// Pluralite, Borda, Copeland. Trois regles, trois facon d'agreger les preferences.

static List<string> PluralityRule(List<List<string>> profile, List<string> alternatives)
{
    // 1 point par premier choix
    var scores = alternatives.ToDictionary(a => a, a => profile.Count(p => p[0] == a));
    return scores.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).Select(kv => kv.Key).ToList();
}

static Dictionary<string, int> BordaScores(List<List<string>> profile, List<string> alternatives)
{
    // n-1 points pour le 1er, 0 pour le dernier
    int n = profile[0].Count;
    var scores = alternatives.ToDictionary(a => a, a => 0);
    foreach (var pref in profile)
        for (int rank = 0; rank < pref.Count; rank++)
            scores[pref[rank]] += (n - 1 - rank);
    return scores;
}

static List<string> BordaRule(List<List<string>> profile, List<string> alternatives)
{
    var scores = BordaScores(profile, alternatives);
    return scores.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).Select(kv => kv.Key).ToList();
}

static Dictionary<string, int> CopelandScores(List<List<string>> profile, List<string> alternatives)
{
    // score = victoires - defaites (pairwise)
    var scores = alternatives.ToDictionary(a => a, a => 0);
    for (int i = 0; i < alternatives.Count; i++)
        for (int j = i + 1; j < alternatives.Count; j++)
        {
            string x = alternatives[i], y = alternatives[j];
            string w = Vote.PairwiseMajority(profile, x, y);
            if (w == x) { scores[x]++; scores[y]--; }
            else if (w == y) { scores[y]++; scores[x]--; }
        }
    return scores;
}

static List<string> CopelandRule(List<List<string>> profile, List<string> alternatives)
{
    var scores = CopelandScores(profile, alternatives);
    return scores.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).Select(kv => kv.Key).ToList();
}

var alts3 = new List<string> { "A", "B", "C" };
Console.WriteLine("COMPARAISON DES REGLES (profil de Condorcet)");
Console.WriteLine(new string('=', 44));
Console.WriteLine($"  Pluralite: {string.Join(" > ", PluralityRule(condorcetProfile, alts3))}");
Console.WriteLine($"  Borda    : {string.Join(" > ", BordaRule(condorcetProfile, alts3))}  (A=B=C=3, ex aequo)");
Console.WriteLine($"  Copeland : {string.Join(" > ", CopelandRule(condorcetProfile, alts3))}  (A=B=C=0, ex aequo)");
Console.WriteLine("\n  Scores Borda : " + string.Join(", ",
    BordaScores(condorcetProfile, alts3).Select(kv => $"{kv.Key}={kv.Value}")));
Console.WriteLine("  Scores Copeland : " + string.Join(", ",
    CopelandScores(condorcetProfile, alts3).Select(kv => $"{kv.Key}={kv.Value}")));


COMPARAISON DES REGLES (profil de Condorcet)


  Pluralite: A > B > C


  Borda    : A > B > C  (A=B=C=3, ex aequo)


  Copeland : A > B > C  (A=B=C=0, ex aequo)



  Scores Borda : A=3, B=3, C=3


  Scores Copeland : A=0, B=0, C=0


## 3. Regles de vote positionnelles

Trois regles, trois logiques d'agregation :
- **Pluralite** : 1 point par premier choix
- **Borda** : n-1 points pour le 1er, ... , 0 pour le dernier
- **Copeland** : +1 par victoire pairwise, -1 par defaite

Sur le profil de Condorcet (cycle parfait), les trois regles donnent des ex aequo (symetrie totale).

In [6]:
// === Section 4 : Profil a resultats divergents ===
// Un meme profil ou Pluralite, Borda et Copeland donnent des gagnants differents.

var divergentProfile = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "A", "B", "C" },
    new() { "B", "C", "A" },
    new() { "C", "B", "A" },
    new() { "C", "B", "A" },
};
PrintProfile("PROFIL AVEC RESULTATS DIVERGENTS", divergentProfile);

var bordaDiv = BordaScores(divergentProfile, alts3);
var copelandDiv = CopelandScores(divergentProfile, alts3);
Console.WriteLine($"\n  Pluralite: {string.Join(" > ", PluralityRule(divergentProfile, alts3))}  (A=2, C=2, B=1)");
Console.WriteLine($"  Borda    : {string.Join(" > ", BordaRule(divergentProfile, alts3))}  ({string.Join(", ", bordaDiv.Select(kv => $"{kv.Key}={kv.Value}"))})");
Console.WriteLine($"  Copeland : {string.Join(" > ", CopelandRule(divergentProfile, alts3))}  ({string.Join(", ", copelandDiv.Select(kv => $"{kv.Key}={kv.Value}"))})");
Console.WriteLine("\n=> Meme profil, resultats differents selon la regle ! (B gagne selon Borda et Copeland, ex aequo A=C en pluralite)");


PROFIL AVEC RESULTATS DIVERGENTS


  Votant 1: A > B > C


  Votant 2: A > B > C


  Votant 3: B > C > A


  Votant 4: C > B > A


  Votant 5: C > B > A



  Pluralite: A > C > B  (A=2, C=2, B=1)


  Borda    : B > C > A  (A=4, B=6, C=5)


  Copeland : B > C > A  (A=-2, B=2, C=0)



=> Meme profil, resultats differents selon la regle ! (B gagne selon Borda et Copeland, ex aequo A=C en pluralite)


## 4. Divergence des regles

Sur un profil ou les preferences sont **asymetriques**, les trois regles donnent des gagnants differents. C'est l'illustration concrete qu'aucune regle n'est "naturelle" -- le choix de la regle determine le vainqueur.

In [7]:
// === Section 5 : Theoreme de Sen (1970) -- Liberte vs Pareto ===
// L'exemple de Lady Chatterley : conflit entre liberte individuelle et efficacite Pareto.
// Deterministe (argument logique, pas de calcul aleatoire).

// np = personne ne lit, pr = Prude lit, lr = Lewd lit
var prudePref = new List<string> { "np", "pr", "lr" };   // Prude : np > pr > lr
var lewdPref  = new List<string> { "pr", "lr", "np" };   // Lewd  : pr > lr > np
var senProfile = new List<List<string>> { prudePref, lewdPref };

Console.WriteLine("L'EXEMPLE DE LADY CHATTERLEY (Theoreme de Sen)");
Console.WriteLine(new string('=', 50));
Console.WriteLine("  Alternatives : np = personne ne lit, pr = Prude lit, lr = Lewd lit");
Console.WriteLine($"  Preferences Prude : {string.Join(" > ", prudePref)}");
Console.WriteLine($"  Preferences Lewd  : {string.Join(" > ", lewdPref)}");

Console.WriteLine("\n--- Principe de liberte minimale ---");
Console.WriteLine("  Prude decide entre pr et np  => socialement np > pr");
Console.WriteLine("  Lewd decide entre lr et np   => socialement lr > np");
Console.WriteLine("\n--- Par transitivite ---");
Console.WriteLine("  lr > np et np > pr  =>  lr > pr");
Console.WriteLine("\n--- Principe de Pareto (unanimite) ---");
bool paretoPrLr = Vote.Prefers(senProfile, 0, "pr", "lr") && Vote.Prefers(senProfile, 1, "pr", "lr");
Console.WriteLine($"  Prude prefere pr > lr ? {Vote.Prefers(senProfile, 0, "pr", "lr")}");
Console.WriteLine($"  Lewd  prefere pr > lr ? {Vote.Prefers(senProfile, 1, "pr", "lr")}");
Console.WriteLine($"  => Pareto(pr, lr) = {paretoPrLr}  => socialement pr > lr");
Console.WriteLine("\n*** CONTRADICTION ***");
Console.WriteLine("  Liberte + Transitivite => lr > pr");
Console.WriteLine("  Pareto                 => pr > lr");
Console.WriteLine("  => Liberte minimale et Pareto sont INCOMPATIBLES (Sen 1970)");


L'EXEMPLE DE LADY CHATTERLEY (Theoreme de Sen)


  Alternatives : np = personne ne lit, pr = Prude lit, lr = Lewd lit


  Preferences Prude : np > pr > lr


  Preferences Lewd  : pr > lr > np



--- Principe de liberte minimale ---


...

  => Pareto(pr, lr) = True  => socialement pr > lr



*** CONTRADICTION ***


  Liberte + Transitivite => lr > pr


  Pareto                 => pr > lr


  => Liberte minimale et Pareto sont INCOMPATIBLES (Sen 1970)


## 5. Theoreme de Sen (1970) : Liberte vs Pareto

L'exemple de Lady Chatterley : le **principe de liberte minimale** (chacun decide sur sa sphere privee) combine a la **transitivite** et au **principe de Pareto** mene a une contradiction. Sen prouve ainsi que liberte et Pareto sont incompatibles. Deterministe (argument logique).

In [8]:
// === Section 6 : Theoreme d'Arrow -- demonstration deterministe d'une violation de IIA ===
// IIA (Independance of Irrelevant Alternatives) : le classement social entre A et B ne
// doit dependre QUE des preferences individuelles entre A et B (pas de la position de C).
// On construit DEUX profils ou chaque electeur a la MEME preference A-vs-B, mais ou le
// classement Borda de A vs B s'INVERSE -> Borda viole IIA. Demo deterministe (parite exacte
// avec le raisonnement ; le twin Python utilise une simulation stochastique equivalente).

// Profil 1 : 3 electeurs A>B>C, 2 electeurs B>C>A  (A>B pour 3, B>A pour 2)
var iiaProfil1 = new List<List<string>>
{
    new() { "A", "B", "C" }, new() { "A", "B", "C" }, new() { "A", "B", "C" },
    new() { "B", "C", "A" }, new() { "B", "C", "A" },
};
// Profil 2 : 3 electeurs A>C>B, 2 electeurs B>A>C  (A>B pour 3, B>A pour 2 -- IDENTIQUE)
var iiaProfil2 = new List<List<string>>
{
    new() { "A", "C", "B" }, new() { "A", "C", "B" }, new() { "A", "C", "B" },
    new() { "B", "A", "C" }, new() { "B", "A", "C" },
};

Console.WriteLine("DEMONSTRATION DETERMINISTE : Borda viole IIA");
Console.WriteLine(new string('=', 50));
Console.WriteLine("  Preferences individuelles A-vs-B IDENTIQUES dans les deux profils :");
Console.WriteLine("    Profil 1 : 3 votants A>B, 2 votants B>A");
Console.WriteLine("    Profil 2 : 3 votants A>B, 2 votants B>A  (identique)");

var b1 = BordaScores(iiaProfil1, alts3);
var b2 = BordaScores(iiaProfil2, alts3);
Console.WriteLine($"\n  Profil 1 (C en derniere position) : Borda A={b1["A"]}, B={b1["B"]}, C={b1["C"]}");
string winner1 = b1["A"] > b1["B"] ? "A" : (b1["B"] > b1["A"] ? "B" : "ex aequo");
Console.WriteLine($"    => Borda : {(b1["A"] >= b1["B"] ? "A" : "B")} au-dessus de {(b1["A"] >= b1["B"] ? "B" : "A")} (B={b1["B"]} > A={b1["A"]})");
Console.WriteLine($"  Profil 2 (C au milieu)            : Borda A={b2["A"]}, B={b2["B"]}, C={b2["C"]}");
Console.WriteLine($"    => Borda : {(b2["A"] >= b2["B"] ? "A" : "B")} au-dessus de {(b2["A"] >= b2["B"] ? "B" : "A")} (A={b2["A"]} > B={b2["B"]})");
Console.WriteLine("\n  *** VIOLATION DE IIA ***");
Console.WriteLine("  Les preferences A-vs-B sont identiques, yet Borda inverse son classement A/B");
Console.WriteLine("  quand C (irrelevant) passe de la derniere a la position mediane.");
Console.WriteLine("  => Borda ne satisfait pas IIA (theoreme d'Arrow, 1951).");


DEMONSTRATION DETERMINISTE : Borda viole IIA


  Preferences individuelles A-vs-B IDENTIQUES dans les deux profils :


    Profil 1 : 3 votants A>B, 2 votants B>A


    Profil 2 : 3 votants A>B, 2 votants B>A  (identique)



  Profil 1 (C en derniere position) : Borda A=6, B=7, C=2


...

    => Borda : A au-dessus de B (A=8 > B=4)



  *** VIOLATION DE IIA ***


  Les preferences A-vs-B sont identiques, yet Borda inverse son classement A/B


  quand C (irrelevant) passe de la derniere a la position mediane.


  => Borda ne satisfait pas IIA (theoreme d'Arrow, 1951).


## 6. Theoreme d'Arrow : demonstration d'une violation de IIA

Le **theoreme d'Arrow (1951)** : aucune regle d'agregation (avec >= 3 alternatives) ne satisfait simultanement Universalite, Pareto, IIA et Non-dictature. On le demontre ici de facon **deterministe** (parite exacte) : deux profils ou chaque electeur a la meme preference A-vs-B, mais ou le classement Borda de A vs B s'inverse quand l'alternative irrelevant C est deplacee. Le twin Python utilise une simulation stochastique (random.shuffle) qui arrive a la meme conclusion ; ce twin C# prefere une demo deterministe reproductible.

In [9]:
// === Section 7 : Theoreme de l'electeur median (Black, Duncan) ===
// Avec des preferences unimodales (single-peaked), le gagnant de Condorcet existe
// toujours et correspond au pic de l'electeur median. Deterministe.

static List<int> SinglePeakedPreference(int peak, List<int> alternatives)
    => alternatives.OrderBy(a => Math.Abs(a - peak)).ToList();

static int FindMedianVoter(List<int> peaks)
{
    var sorted = peaks.OrderBy(p => p).ToList();
    return sorted[sorted.Count / 2];
}

var alternativesSp = new List<int> { 0, 2, 4, 6, 8, 10 };
var voterPeaks = new List<int> { 1, 3, 4, 5, 7, 8, 9 };   // 7 electeurs

Console.WriteLine("THEOREME DE L'ELECTEUR MEDIAN");
Console.WriteLine(new string('=', 40));
Console.WriteLine($"  Alternatives : [{string.Join(", ", alternativesSp)}]");
Console.WriteLine($"  Pics des electeurs : [{string.Join(", ", voterPeaks)}]");
Console.WriteLine("\n  Preferences generees (unimodales) :");
var profileSp = voterPeaks.Select(p => SinglePeakedPreference(p, alternativesSp)).ToList();
for (int i = 0; i < voterPeaks.Count; i++)
    Console.WriteLine($"    Electeur {i + 1} (pic={voterPeaks[i]}): [{string.Join(", ", profileSp[i])}]");

int medianPeak = FindMedianVoter(voterPeaks);
Console.WriteLine($"\n  Pic median : {medianPeak}");
// Gagnant de Condorcet = alternative la plus proche du pic median
int winnerSp = alternativesSp.OrderBy(a => Math.Abs(a - medianPeak)).First();
Console.WriteLine($"  Gagnant de Condorcet (single-peaked) : {winnerSp}");
Console.WriteLine("  => Avec des preferences unimodales, PAS de cycle de Condorcet.");


THEOREME DE L'ELECTEUR MEDIAN


  Alternatives : [0, 2, 4, 6, 8, 10]


  Pics des electeurs : [1, 3, 4, 5, 7, 8, 9]



  Preferences generees (unimodales) :


    Electeur 1 (pic=1): [0, 2, 4, 6, 8, 10]


...

    Electeur 6 (pic=8): [8, 6, 10, 4, 2, 0]


    Electeur 7 (pic=9): [8, 10, 6, 4, 2, 0]



  Pic median : 5


  Gagnant de Condorcet (single-peaked) : 4


  => Avec des preferences unimodales, PAS de cycle de Condorcet.


## 7. Theoreme de l'electeur median (Black)

Avec des **preferences unimodales** (single-peaked), le gagnant de Condorcet existe toujours et correspond au pic de l'electeur median. Le cycle de Condorcet disparait. Deterministe.

In [10]:
// === Section 8 : Convergence vers le centre (modele de Downs) ===
// ATTENTION (parite) : le twin Python tire des electeurs via np.random.normal(5,2,100)
// avec seed 42 (Mersenne Twister). System.Random en C# (seed 42) produit une distribution
// differente. On simule donc la convergence avec une grille DETERMINISTE d'electeurs
// (7 pics fixes, meme principe) pour obtenir une parite exacte sur la trajectoire.
// Le RESULTAT pedagogique (les deux partis convergent vers le median) est preserve.

static (List<double> histL, List<double> histR, double median) SimulateTwoParty(
    List<int> peaks, int nRounds)
{
    double partyL = 2.0, partyR = 8.0;
    var histL = new List<double> { partyL };
    var histR = new List<double> { partyR };
    double median = peaks.OrderBy(p => p).ToList()[peaks.Count / 2];
    for (int r = 0; r < nRounds; r++)
    {
        if (partyL < median) partyL = Math.Min(partyL + 0.3, median);
        if (partyR > median) partyR = Math.Max(partyR - 0.3, median);
        histL.Add(partyL);
        histR.Add(partyR);
    }
    return (histL, histR, median);
}

// Pics deterministes (grille) -- evite le tirage aleatoire cross-langage
var peaksDet = new List<int> { 1, 2, 3, 4, 5, 6, 7, 8, 9 };
var (histL, histR, medianD) = SimulateTwoParty(peaksDet, nRounds: 20);

Console.WriteLine("MODELE DE DOWNS : convergence vers le centre (2 partis)");
Console.WriteLine(new string('=', 54));
Console.WriteLine($"  Median des electeurs : {medianD:F1}");
Console.WriteLine($"  Parti de gauche : {histL[0]:F1} -> {histL[^1]:F1}  (converge vers le median)");
Console.WriteLine($"  Parti de droite  : {histR[0]:F1} -> {histR[^1]:F1}  (converge vers le median)");
// Visualisation ASCII de la convergence
Console.WriteLine("\n  Trajectoire (G=gauche, D=droite, |=median) :");
for (int t = 0; t < histL.Count; t += 4)
    Console.WriteLine($"    t={t,2}: G={histL[t]:F1}   D={histR[t]:F1}");
Console.WriteLine($"\n  => Les deux partis convergent vers la position mediane ({medianD:F1}).");


MODELE DE DOWNS : convergence vers le centre (2 partis)


  Median des electeurs : 5.0


  Parti de gauche : 2.0 -> 5.0  (converge vers le median)


  Parti de droite  : 8.0 -> 5.0  (converge vers le median)



  Trajectoire (G=gauche, D=droite, |=median) :


...

    t= 8: G=4.4   D=5.6


    t=12: G=5.0   D=5.0


    t=16: G=5.0   D=5.0


    t=20: G=5.0   D=5.0



  => Les deux partis convergent vers la position mediane (5.0).


## 8. Modele de Downs : convergence vers le centre

Deux partis en competition ajustent leur position pour maximiser leurs votes ; ils convergent tous deux vers la position mediane (theoreme de l'electeur median applique a la competition partisane).

> **Note de parite** : le twin Python tire 100 electeurs via `np.random.normal(5,2)` (seed 42). Pour une parite exacte de la trajectoire, ce twin C# utilise une **grille deterministe** de pics. Le resultat (convergence vers le median) est identique.

In [11]:
// === Section 9 : Avec 2 alternatives, la majorite fonctionne ===
// Le theoreme d'Arrow requiert |A| >= 3. Avec 2 alternatives, pas de cycle possible.

Console.WriteLine("AVEC 2 ALTERNATIVES : LA MAJORITE FONCTIONNE");
Console.WriteLine(new string('=', 50));
var profile2alt = new List<List<string>>
{
    new() { "A", "B" },
    new() { "A", "B" },
    new() { "B", "A" },
};
PrintProfile("  Exemple (3 electeurs)", profile2alt);
string winner2 = Vote.PairwiseMajority(profile2alt, "A", "B");
Console.WriteLine($"\n  Resultat majoritaire : {winner2} gagne (2 contre 1)");
Console.WriteLine("  => Pas de cycle possible avec 2 alternatives (relation complete et antisymetrique).");
Console.WriteLine("     Le theoreme d'Arrow (dictateur ou violation d'une autre propriete) requiert >= 3 alternatives.");


AVEC 2 ALTERNATIVES : LA MAJORITE FONCTIONNE


  Exemple (3 electeurs)


  Votant 1: A > B


  Votant 2: A > B


  Votant 3: B > A



  Resultat majoritaire : A gagne (2 contre 1)


  => Pas de cycle possible avec 2 alternatives (relation complete et antisymetrique).


     Le theoreme d'Arrow (dictateur ou violation d'une autre propriete) requiert >= 3 alternatives.


## 9. Avec 2 alternatives, la majorite fonctionne

Le theoreme d'Arrow requiert **|A| >= 3**. Avec 2 alternatives, le vote majoritaire satisfait Pareto, IIA et Non-dictature : pas de cycle possible.

In [12]:
// === Section 10 : Exercices (stubs C.1 -- a completer par l'etudiant) ===
// Exercice 1 : Gagnant de Condorcet et cycles. Profil a 4 candidats.
static string TrouverGagnantCondorcet(List<List<string>> profil, List<string> candidats)   // TODO etudiant
{
    // Indice : pour chaque candidat, verifier s'il bat tous les autres en duel pairwise
    // (reutiliser CondorcetWinner ci-dessus, ou le reimplementer).
    // Etape 1 : parcourir chaque candidat.
    // Etape 2 : pour chacun, verifier tous les duels.
    // Etape 3 : retourner le gagnant, ou null s'il y a un cycle.
    Console.WriteLine("Exercice a completer : gagnant de Condorcet sur un profil a 4 candidats");
    return null;
}

var profil1 = new List<List<string>>
{
    new() { "A", "B", "D", "C" },
    new() { "B", "C", "A", "D" },
    new() { "C", "D", "A", "B" },
    new() { "D", "A", "B", "C" },
    new() { "A", "C", "D", "B" },
};
TrouverGagnantCondorcet(profil1, new() { "A", "B", "C", "D" });


Exercice a completer : gagnant de Condorcet sur un profil a 4 candidats


In [13]:
// Exercice 2 : Vote par elimination successive (IRV / instant-runoff).
static List<string> InstantRunoffVoting(List<List<string>> profil, List<string> candidats)   // TODO etudiant
{
    // Indice : a chaque tour, compter les premiers choix. Si un candidat a la majorite
    // absolue (> 50%), c'est le gagnant. Sinon, eliminer le candidat avec le moins de
    // premiers choix et recommencer.
    // Etape 1 : boucle sur les tours (tant qu'aucun majorite absolue).
    // Etape 2 : compter les premiers choix parmi les candidats restants.
    // Etape 3 : eliminer le dernier et continuer.
    Console.WriteLine("Exercice a completer : vote par elimination successive (IRV)");
    return candidats;
}

var profilA = new List<List<string>>
{
    new() { "X", "Y", "Z" }, new() { "X", "Y", "Z" }, new() { "X", "Y", "Z" }, new() { "X", "Y", "Z" },
    new() { "Y", "Z", "X" }, new() { "Y", "Z", "X" }, new() { "Y", "Z", "X" },
    new() { "Z", "Y", "X" }, new() { "Z", "Y", "X" },
};
InstantRunoffVoting(profilA, new() { "X", "Y", "Z" });


Exercice a completer : vote par elimination successive (IRV)


In [14]:
// Exercice 3 : Theoreme d'Arrow -- verifier IIA pour Borda sur deux profils.
static bool VerifierIiaBorda(List<List<string>> profil1, List<List<string>> profil2,
    List<string> candidats, string x, string y)   // TODO etudiant
{
    // Indice : IIA dit que le classement SOCIAL entre x et y ne doit dependre que des
    // preferences individuelles entre x et y. Si profil1 et profil2 ont les memes preferences
    // x-vs-y pour tous les electeurs mais un classement social x/y different, IIA est violee.
    // Etape 1 : verifier que les preferences x-vs-y sont identiques entre profil1 et profil2.
    // Etape 2 : calculer le classement Borda de chacun.
    // Etape 3 : comparer la position relative de x et y ; retourner true si IIA respectee.
    Console.WriteLine("Exercice a completer : verification IIA pour Borda");
    return true;
}

var profilIia1 = new List<List<string>>
{
    new() { "A", "B", "C" }, new() { "B", "A", "C" }, new() { "A", "C", "B" },
};
var profilIia2 = new List<List<string>>
{
    new() { "A", "B", "C" }, new() { "B", "A", "C" }, new() { "C", "A", "B" },
};
VerifierIiaBorda(profilIia1, profilIia2, new() { "A", "B", "C" }, "A", "B");

Console.WriteLine("\n=== Recapitulatif ===");
Console.WriteLine("Aucune regle d'agregation (avec >= 3 alternatives) ne satisfait simultanement");
Console.WriteLine("Universalite, Pareto, IIA et Non-dictature (theoreme d'Arrow, 1951).");
Console.WriteLine("Les methodes positionnelles (Pluralite, Borda) et pairwise (Copeland, Condorcet)");
Console.WriteLine("offrent des compromis differents. Les preuves formelles vivent dans le notebook");
Console.WriteLine("Lean SC-02 ; ce twin C# execute les algorithmes from-scratch (Prong B).");


Exercice a completer : verification IIA pour Borda



=== Recapitulatif ===


Aucune regle d'agregation (avec >= 3 alternatives) ne satisfait simultanement


Universalite, Pareto, IIA et Non-dictature (theoreme d'Arrow, 1951).


Les methodes positionnelles (Pluralite, Borda) et pairwise (Copeland, Condorcet)


offrent des compromis differents. Les preuves formelles vivent dans le notebook


Lean SC-02 ; ce twin C# execute les algorithmes from-scratch (Prong B).


## 10. Exercices

Trois exercices en C# (stubs `TODO etudiant`, regle C.1 -- aucune erreur volontaire, le notebook s'execute de bout en bout).